# SignBridge AI — Step 4: Verify Dataset

Verify dataset integrity, check for missing/corrupt files, and compute statistics.

In [ ]:
import os
import csv
import json
from pathlib import Path

DATASET_DIR = '/content/drive/MyDrive/SignBridgeAI/dataset'
CSV_PATH = os.path.join(DATASET_DIR, 'iSign_v1.1.csv')
POSE_DIR = os.path.join(DATASET_DIR, 'poses')
VIDEO_DIR = os.path.join(DATASET_DIR, 'videos')

In [ ]:
# Verify CSV
print('=== CSV Verification ===')
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        print(f'  Rows: {len(rows)}')
        print(f'  Columns: {reader.fieldnames}')
        print(f'  Sample: {rows[0]}')
else:
    print(f'  CSV not found at {CSV_PATH}')

In [ ]:
# Verify poses
print('=== Pose Verification ===')
if os.path.exists(POSE_DIR):
    pose_files = list(Path(POSE_DIR).rglob('*.npy'))
    print(f'  Total pose files: {len(pose_files)}')
    
    # Check a sample
    if pose_files:
        import numpy as np
        sample = np.load(str(pose_files[0]))
        print(f'  Sample shape: {sample.shape}')
        print(f'  Sample dtype: {sample.dtype}')
else:
    print(f'  Pose directory not found at {POSE_DIR}')

In [ ]:
# Check UIDs match
print('=== UID Matching ===')
csv_uids = set()
with open(CSV_PATH, 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        uid = row['uid']
        base = uid.split('_')[0] if '_' in uid else uid
        csv_uids.add(base)

pose_uids = set(f.stem for f in Path(POSE_DIR).rglob('*.npy')) if os.path.exists(POSE_DIR) else set()

matched = csv_uids & pose_uids
missing = csv_uids - pose_uids
extra = pose_uids - csv_uids

print(f'  CSV UIDs: {len(csv_uids)}')
print(f'  Pose UIDs: {len(pose_uids)}')
print(f'  Matched: {len(matched)}')
print(f'  Missing poses: {len(missing)}')
print(f'  Extra poses: {len(extra)}')

In [ ]:
# Statistics
print('=== Dataset Statistics ===')
text_lengths = []
vocab = set()
with open(CSV_PATH, 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        text = row['text']
        text_lengths.append(len(text.split()))
        vocab.update(text.lower().split())

print(f'  Total rows: {len(text_lengths)}')
print(f'  Unique videos: {len(csv_uids)}')
print(f'  Vocab size: {len(vocab)}')
print(f'  Avg text length: {sum(text_lengths)/len(text_lengths):.1f} words')
print(f'  Max text length: {max(text_lengths)} words')

# Storage
total = 0
for d in [POSE_DIR, VIDEO_DIR]:
    if os.path.exists(d):
        for f in Path(d).rglob('*'):
            if f.is_file():
                total += f.stat().st_size
print(f'  Storage used: {total/1024**3:.2f} GB')